In [ ]:
import pandas as pd
import numpy as np
import os
import itertools
import time
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from io import BytesIO
from scipy.stats import kstest, norm, stats, probplot
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Attention, Concatenate, Input, Layer
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import Huber
from tensorflow.keras.optimizers import Adam
import shap
from tensorflow.keras.layers import Lambda
from tensorflow.keras.models import Model
from xgboost import XGBRegressor
import xgboost as xgb
from tensorflow.keras import mixed_precision
import joblib
from sklearn.inspection import permutation_importance
import matplotlib.dates as mdates

In [ ]:
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# —— 一次性设置，全局生效（期刊单栏常用字号）——
RC = {
    'font.size': 9,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
    'axes.unicode_minus': False
}
sns.set_style("whitegrid")     # 或 "white"
FIGSIZE = (6.8, 3.5)   # 两张图都用同一尺寸（建议）
SAVE_KW = dict(bbox_inches='tight', pad_inches=0.02)  # 两张图保存方式一致
os.makedirs("Figures", exist_ok=True)

# Data Analysis


In [ ]:
df = pd.read_csv('cork_2012_2024_6h.csv')  
df = df.drop(columns=['number', 'expver'])
df.head(100)

In [ ]:


# 提取特征和目标
features = ['u10', 'v10', 'd2m', 't2m', 'sp', 'tcc', 'lsp_mm']
X = df[features]
y = df['tp_mm']

# Z-score 标准化
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 保存均值与标准差以便反标准化
mean_tp_mm = y.mean()
std_tp_mm = y.std()
y_scaled = (y - mean_tp_mm) / std_tp_mm

In [ ]:
df.head(100)

In [ ]:
# 缺失值填充
df.loc[:, 'tp_mm'] = df['tp_mm'].ffill()  # 前向填充
df['lsp_mm'] = df['lsp_mm'].interpolate()

In [ ]:
# 时间处理
df['time'] = pd.to_datetime(df['valid_time'])

# 选取变量
X_vars = ['u10', 'v10', 'd2m', 't2m', 'sp', 'tcc', 'lsp_mm']
y_var = 'tp_mm'

# 提取包含目标变量的数据
df_to_analyze = df[X_vars + [y_var]]  # y_var='tp_mm'

# 计算 Spearman 相关系数矩阵
spearman_corr = df_to_analyze.corr(method='spearman')
print(spearman_corr)  # 控制台输出完整矩阵（包含 tp_mm）

# 假设已定义 RC / FIGSIZE / SAVE_KW，并已 import seaborn as sns, matp_mmlotlib.pyplot as plt

with plt.rc_context(RC):
    # 若想与其它图同宽，但做成正方形画布：用同样宽度的方形尺寸
    FIGSIZE_SQUARE = (FIGSIZE[0], FIGSIZE[0])   # 例如 (6.8, 6.8)

    fig, ax = plt.subplots(figsize=FIGSIZE_SQUARE)

    sns.heatmap(
        spearman_corr,
        cmap=sns.color_palette("Blues", as_cmap=True),  # 纯蓝渐变
        annot=True, fmt=".2f",
        annot_kws={"size": RC['font.size']},
        vmin=-1, vmax=1,
        linewidths=0,               # 无白框
        square=True,                # 单元格正方形（1:1）
        cbar_kws={"label": "Spearman $\\rho$", "shrink": 0.8, "pad": 0.02},
        ax=ax
    )

    # 去掉轴外框
    for sp in ax.spines.values():
        sp.set_visible(False)

    ax.set_title("Spearman Correlation Heatmap")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    fig.tight_layout()
    fig.savefig("Spearman_heatmap_blue_square.pdf", **SAVE_KW)
    fig.savefig("Spearman_heatmap_blue_square.png", dpi=300, **SAVE_KW)
    plt.close(fig)


In [ ]:
# 时间格式转换
df['time'] = pd.to_datetime(df['valid_time'])

time_all = df['time'].values 
# 筛选后的变量（根据 Spearman 相关性和物理意义）
X_vars = ['lsp_mm', 'tcc', 'v10', 'u10', 'd2m']  # 输入特征
y_var = 'tp_mm'  # 预测目标

# 提取变量
X = df[X_vars].values
y = df[[y_var]].values

# 标准化
from sklearn.preprocessing import StandardScaler
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_x.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# 构造时间序列格式
def create_sequences_with_time(X, y, times, window_size=6, horizon=1, step=1):
    X_seq, y_seq, t_seq = [], [], []
    N = len(X)

    for i in range(0, N - window_size - horizon + 1, step):
        X_seq.append(X[i : i + window_size])
        y_seq.append(y[i + window_size + horizon - 1])
        t_seq.append(times[i + window_size + horizon - 1])  # 对应预测点时间戳
    return np.array(X_seq), np.array(y_seq), np.array(t_seq)
X_seq, y_seq, t_seq = create_sequences_with_time(
    X_scaled, y_scaled, time_all,
    window_size=6, horizon=1, step=1
)


In [ ]:
# 数据 reshape（如果你每个时间步是多特征输入）
# 输入维度: (samples, timesteps, features)
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

In [ ]:
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def call(self, inputs):
        query, value = inputs, inputs
        score = tf.matmul(query, value, transpose_b=True)
        weights = tf.nn.softmax(score, axis=-1)
        context = tf.matmul(weights, value)
        return context


In [ ]:
timesteps = X_train.shape[1]
features = X_train.shape[2]

inputs = Input(shape=(timesteps, features))

# BiLSTM 1
x = Bidirectional(LSTM(
    32,
    return_sequences=True,
    kernel_regularizer=tf.keras.regularizers.l2(1e-4)
))(inputs)
x = Dropout(0.4)(x)

# BiLSTM 2
x = Bidirectional(LSTM(
    16,
    return_sequences=True,
    kernel_regularizer=tf.keras.regularizers.l2(1e-4)
))(x)
x = Dropout(0.4)(x)

# Attention
attention = AttentionLayer()(x)
attention = Lambda(lambda xx: tf.reduce_mean(xx, axis=1))(attention)

mean_outp_mmut = Dense(1)(attention)
log_var_outp_mmut = Dense(1)(attention)

model = Model(inputs, [mean_outp_mmut, log_var_outp_mmut])


In [ ]:
def nll_loss(y_true, y_pred_mean, y_pred_log_var):
    y_true = tf.cast(y_true, tf.float32)
    precision = tf.exp(-y_pred_log_var)
    return tf.reduce_mean(0.5 * precision * tf.square(y_true - y_pred_mean) + 0.5 * y_pred_log_var)

In [ ]:
# 自定义模型类
class CustomNLLModel(tf.keras.Model):
    def __init__(self, model, **kwargs):
        super().__init__(**kwargs)
        self.model = model  # 这是 Functional API 构建的模型

    def call(self, inputs, training=False):
        return self.model(inputs, training=training)

    @tf.function
    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            y_pred_mean, y_pred_log_var = self(x, training=True)
            loss = nll_loss(y, y_pred_mean, y_pred_log_var)

        gradients = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))
        self.compiled_metrics.update_state(y, y_pred_mean)
        return {"loss": loss}

    @tf.function
    def test_step(self, data):
        x, y = data
        y_pred_mean, y_pred_log_var = self(x, training=False)
        loss = nll_loss(y, y_pred_mean, y_pred_log_var)
        self.compiled_metrics.update_state(y, y_pred_mean)
        return {"loss": loss}

# 用你构建好的 mean 和 log_var 模型输出构建一个基础模型
base_model = Model(inputs=inputs,
              outputs=[mean_outp_mmut, log_var_outp_mmut])

# 1. 定义学习率衰减
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.001,
    decay_steps=2000,
    decay_rate=0.96,
    staircase=True
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=5e-5,
    clipnorm=1.0
)

nll_model = CustomNLLModel(model)

nll_model.compile(
    optimizer=optimizer,
    loss=None
)

# 包装成自定义训练模型
nll_model = CustomNLLModel(base_model)
nll_model.compile(optimizer=optimizer, run_eagerly=False)  # 注意：这里不提供 loss

In [ ]:

N = len(X_seq)
train_end = int(N * 0.8)   # 80% 训练
# val_end = int(N * 0.9)  # 如果你之后需要 val，可以加

X_train, X_test = X_seq[:train_end], X_seq[train_end:]
y_train, y_test = y_seq[:train_end], y_seq[train_end:]
t_train, t_test = t_seq[:train_end], t_seq[train_end:]


# Step 2: 构建 tf.data.Dataset 管道（训练集和验证集）
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)) \
                          .shuffle(buffer_size=10000) \
                          .batch(32) \
                          .prefetch(tf.data.AUTOTUNE)

# val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)) \
#                         .batch(32) \
#                         .prefetch(tf.data.AUTOTUNE)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

# Step 3: 使用新的 dataset 来训练模型
history = nll_model.fit(
    train_ds,
    # validation_data=val_ds,
    epochs=200,       # 可以多给一点上限
    callbacks=[early_stop]
)

In [ ]:
# === Step 1: 预测（必须解包） ===
# y_pred_mean, y_pred_log_var = nll_model.predict(X_val)
y_pred = y_pred_mean  # ✅ 使用均值作为预测值

# === Step 2: 可视化前处理 ===
# 保证是 numpy 数组
y_pred = np.array(y_pred)
# y_val = np.array(y_val)

# 如果是三维的，取最后一个时间步
if y_pred.ndim == 3:
    y_pred = y_pred[:, -1, :]
# if y_val.ndim == 3:
#     y_val = y_val[:, -1, :]

# reshape 成二维
y_val = y_val.reshape(-1, 1)
y_pred = y_pred.reshape(-1, 1)

# 截断为相同长度
min_len = min(len(y_pred))
# y_val = y_val[:min_len]
y_pred = y_pred[:min_len]

# 反标准化
# y_val_inv = scaler_y.inverse_transform(y_val)
y_pred_inv = scaler_y.inverse_transform(y_pred)

# ---------- 构造时间轴 ----------
N = len(y_pred_inv)  # 与绘图数据一致
time_axis = None

# try:
#     # 1) 如果 X_val 有 DatetimeIndex
#     if hasattr(X_val, 'index') and isinstance(X_val.index, pd.DatetimeIndex):
#         time_axis = X_val.index[:N]
#     # 2) 如果 X_val 有 'time' 列
#     elif hasattr(X_val, 'columns') and ('time' in X_val.columns):
#         time_axis = pd.to_datetime(X_val['time'].iloc[:N])
# except Exception:
#     pass

# 3) 兜底：用原始 df 的时间（取最后 N 个，假设已按时间排序）
if time_axis is None:
    time_axis = pd.to_datetime(df['time'].iloc[-N:])

# ---------- 可视化（时间刻度） ----------
with plt.rc_context(RC):
    fig, ax = plt.subplots(figsize=FIGSIZE)
    # ax.plot(time_axis, y_val_inv.ravel(),  label='True Values', color='blue', alpha=0.7)
    ax.plot(time_axis, y_pred_inv.ravel(), label='Predictions', color='red', ls='--', alpha=0.7)

    ax.set_xlabel('Time')
    ax.set_ylabel('Target (original scale)')
    ax.set_title('True vs Predicted')
    ax.legend()
    ax.grid(True, ls='--', lw=0.5, alpha=0.7)

    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=30, ha='right')

    fig.tight_layout()
    fig.savefig('Figures/true_vs_pred_time_fixed.pdf', **SAVE_KW)
    fig.savefig('Figures/true_vs_pred_time_fixed.png', dpi=300, **SAVE_KW)
    plt.close(fig)

In [ ]:
# === 直接使用模型输出的 std（已经是原始尺度） ===
std_pred = np.sqrt(np.exp(y_pred_log_var)).reshape(-1, 1)
# 与 y_pred_inv 对齐长度
N = len(y_pred_inv)
std_pred = std_pred[:N]

# === 2) 构造时间轴（优先用 X_val 的索引/列；否则退化到 df['time']） ===
time_axis = None
try:
    if hasattr(X_val, 'index') and isinstance(X_val.index, pd.DatetimeIndex):
        time_axis = X_val.index[:N]
    elif hasattr(X_val, 'columns') and ('time' in X_val.columns):
        time_axis = pd.to_datetime(X_val['time'].iloc[:N])
except Exception:
    pass
if time_axis is None:
    time_axis = pd.to_datetime(df['time'].iloc[-N:])   # 退化方案：用原 df 末 N 个时间

# === 3) 画图（时间横轴 + 95% CI） ===
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(time_axis, y_val_inv.ravel(), label='True Values', color='blue', alpha=0.7)
ax.plot(time_axis, y_pred_inv.ravel(), label='Predictions', color='red', linestyle='--', alpha=0.7)

# 95% 置信区间
lower = y_pred_inv.ravel() - 1.96 * std_pred.ravel()
upper = y_pred_inv.ravel() + 1.96 * std_pred.ravel()
ax.fill_between(time_axis, lower, upper, color='orange', alpha=0.30, label='95% Confidence Interval')

ax.set_xlabel('Time')
ax.set_ylabel('Target (original scale)')
ax.set_title('Prediction with Uncertainty')

# 时间刻度美化
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=30, ha='right')


import os
os.makedirs("Figures", exist_ok=True)

fig, ax = plt.subplots(figsize=FIGSIZE)

# 主曲线
ax.plot(time_axis, y_val_inv.ravel(),  label='True Values',  color='blue', alpha=0.7)
ax.plot(time_axis, y_pred_inv.ravel(), label='Predictions', color='red',  linestyle='--', alpha=0.7)

# 95% CI
lower = y_pred_inv.ravel() - 1.96 * std_pred.ravel()
upper = y_pred_inv.ravel() + 1.96 * std_pred.ravel()
ax.fill_between(time_axis, lower, upper, color='orange', alpha=0.30, label='95% Confidence Interval')

with plt.rc_context(RC):
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(time_axis, y_val_inv.ravel(),  label='True Values', color='blue', alpha=0.7)
    ax.plot(time_axis, y_pred_inv.ravel(), label='Predictions', color='red', ls='--', alpha=0.7)

    lower = y_pred_inv.ravel() - 1.96*std_pred.ravel()
    upper = y_pred_inv.ravel() + 1.96*std_pred.ravel()
    ax.fill_between(time_axis, lower, upper, color='orange', alpha=0.30, label='95% Confidence Interval')

    ax.set_xlabel('Time')
    ax.set_ylabel('Target (original scale)')
    ax.set_title('Prediction with Uncertainty')
    ax.legend()
    ax.grid(True, ls='--', lw=0.5, alpha=0.7)

    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=30, ha='right')

    fig.tight_layout()
    fig.savefig('Figures/pred_with_uncertainty_time.pdf', **SAVE_KW)
    fig.savefig('Figures/pred_with_uncertainty_time.png', dpi=300, **SAVE_KW)
    plt.close(fig)


In [ ]:
# 预测
y_pred_mean_test, y_pred_log_var_test = nll_model.predict(X_test, batch_size=64)

# 反标准化
y_test_inv  = scaler_y.inverse_transform(y_test)
y_pred_inv  = scaler_y.inverse_transform(y_pred_mean_test)

# 标准差
std_pred = np.sqrt(np.exp(y_pred_log_var_test))
if hasattr(scaler_y, 'scale_'):
    std_pred = std_pred * float(np.ravel(scaler_y.scale_)[0])

# 时间轴 = 每个预测点对应的真实时间
time_axis = pd.to_datetime(t_test)

mask_2023 = (time_axis >= '2023-05-01') & (time_axis < '2023-12-01')

time_2023   = time_axis[mask_2023]
y_true_2023 = y_test_inv.ravel()[mask_2023]
y_pred_2023 = y_pred_inv.ravel()[mask_2023]
std_2023    = std_pred.ravel()[mask_2023]

lower_2023 = y_pred_2023 - 1.96 * std_2023
upper_2023 = y_pred_2023 + 1.96 * std_2023

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# 为了图更好看，对 2023 年真实值做 1%~99% 分位裁剪
q_low_2023, q_high_2023 = np.percentile(y_true_2023, [1, 99])

fig, ax = plt.subplots(figsize=(12, 4), dpi=150)

# 置信区间（先画在底层）
ax.fill_between(
    time_2023,
    lower_2023,
    upper_2023,
    color='orange',
    alpha=0.30,
    label='95% Confidence Interval'
)

# 真实值
ax.plot(
    time_2023,
    y_true_2023,
    label='True Values',
    color='blue',
    alpha=0.8,
    linewidth=0.8
)

# 预测均值
ax.plot(
    time_2023,
    y_pred_2023,
    label='Predictions',
    color='red',
    linestyle='-',
    alpha=0.9,
    linewidth=1.2
)

# y 轴范围按分位数裁剪，避免极端值把图压扁
ax.set_ylim(q_low_2023, q_high_2023)

# 时间坐标：每月一个 tick
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=30, ha='right')

ax.set_title('Prediction with Uncertainty')
ax.set_xlabel('Time')
ax.set_ylabel('Target (original scale)')
ax.grid(True, ls='--', lw=0.5, alpha=0.7)
ax.legend(loc='upper right', bbox_to_anchor=(1.02, 1.0), borderaxespad=0.)


plt.tight_layout()

import os
os.makedirs('Figures', exist_ok=True)
fig.savefig('Figures/pred_with_uncertainty_2023.pdf')
fig.savefig('Figures/pred_with_uncertainty_2023.png', dpi=300)

plt.show()


In [ ]:
# === 预测 ===
y_pred_mean_test, y_pred_log_var_test = nll_model.predict(X_test, batch_size=64)

# 反标准化
y_test_inv  = scaler_y.inverse_transform(y_test)
y_pred_inv  = scaler_y.inverse_transform(y_pred_mean_test)

# 标准差
std_pred = np.sqrt(np.exp(y_pred_log_var_test))
if hasattr(scaler_y, 'scale_'):
    std_pred = std_pred * float(np.ravel(scaler_y.scale_)[0])

# 时间轴：对应预测点的真实时间
time_axis = pd.to_datetime(t_test)

# === ★ 截取 2023 年 11 月整月（关键修改）★ ===
mask_2023_11 = (time_axis >= '2023-11-01') & (time_axis < '2023-12-01')

time_nov   = time_axis[mask_2023_11]
y_true_nov = y_test_inv.ravel()[mask_2023_11]
y_pred_nov = y_pred_inv.ravel()[mask_2023_11]
std_nov    = std_pred.ravel()[mask_2023_11]

lower_nov = y_pred_nov - 1.96 * std_nov
upper_nov = y_pred_nov + 1.96 * std_nov


# === 画图 ===
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# y 轴按分位数裁剪（保证可视化更好）
q_low, q_high = np.percentile(y_true_nov, [1, 99])

fig, ax = plt.subplots(figsize=(12, 4), dpi=150)

# 置信区间
ax.fill_between(
    time_nov,
    lower_nov,
    upper_nov,
    color='orange',
    alpha=0.30,
    label='95% Confidence Interval'
)

# True
ax.plot(
    time_nov,
    y_true_nov,
    label='True Values',
    color='blue',
    alpha=0.8,
    linewidth=0.8
)

# Pred
ax.plot(
    time_nov,
    y_pred_nov,
    label='Predictions',
    color='red',
    linestyle='--',
    alpha=0.9,
    linewidth=1.2
)

# y 轴范围
ax.set_ylim(q_low, q_high)

# 时间刻度：每天一个 tick
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=30, ha='right')

ax.set_title('Prediction with Uncertainty – November 2023')
ax.set_xlabel('Time')
ax.set_ylabel('Target (original scale)')
ax.grid(True, ls='--', lw=0.5, alpha=0.7)
ax.legend(loc='upper left')

plt.tight_layout()

import os
os.makedirs('Figures', exist_ok=True)
fig.savefig('Figures/pred_with_uncertainty_2023_11.pdf')
fig.savefig('Figures/pred_with_uncertainty_2023_11.png', dpi=300)

plt.show()


In [ ]:
# === 预测三组数据（输出均值和 log 方差） ===
y_pred_mean_train, _ = nll_model.predict(X_tr)
y_pred_mean_val, _ = nll_model.predict(X_val)
y_pred_mean_test, y_pred_log_var = nll_model.predict(X_test)

In [ ]:
train_rmse_mm, train_r2 = eval_mm(y_tr,  y_pred_mean_train, "Train")
val_rmse_mm,   val_r2   = eval_mm(y_val, y_pred_mean_val,   "Val")
test_rmse_mm,  test_r2  = eval_mm(y_test, y_pred_mean_test, "Test")


# XGBoost

In [ ]:
# ===== 1) 展平：把 [samples, 6, n_features] -> [samples, 6*n_features] =====
def flatten_3d(arr3d):
    return arr3d.reshape(arr3d.shape[0], -1)

# 如果你已经有 X_tr, X_val, X_test / y_tr, y_val, y_test（和神经网络一致），用它们：
X_tr_flat   = flatten_3d(X_tr)
X_val_flat  = flatten_3d(X_val)
X_test_flat = flatten_3d(X_test)

y_tr_flat   = y_tr.reshape(-1)
y_val_flat  = y_val.reshape(-1)
y_test_flat = y_test.reshape(-1)

# ===== 2) 定义并训练：带验证集早停 =====
xgb_model = xgb.XGBRegressor(
    n_estimators=200,          # 稍多一些，让早停来裁剪
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,            # L2 正则，稳一点
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_tr_flat, y_tr_flat,
    eval_set=[(X_val_flat, y_val_flat)],
    eval_metric="rmse",
    early_stopping_rounds=50,
    verbose=False
)

# ===== 3) 预测（会自动用 best_iteration_）=====
y_pred_tr   = xgb_model.predict(X_tr_flat)
y_pred_val  = xgb_model.predict(X_val_flat)
y_pred_test = xgb_model.predict(X_test_flat)

# ===== 4) 反标准化到 mm/6h（与 BU-LSTM 一致，不再 /1000）=====
y_tr_mm       = scaler_y.inverse_transform(y_tr_flat.reshape(-1,1)).ravel()
y_val_mm      = scaler_y.inverse_transform(y_val_flat.reshape(-1,1)).ravel()
y_test_mm     = scaler_y.inverse_transform(y_test_flat.reshape(-1,1)).ravel()

y_pred_tr_mm   = scaler_y.inverse_transform(y_pred_tr.reshape(-1,1)).ravel()
y_pred_val_mm  = scaler_y.inverse_transform(y_pred_val.reshape(-1,1)).ravel()
y_pred_test_mm = scaler_y.inverse_transform(y_pred_test.reshape(-1,1)).ravel()

# 如需用“米/6h”，统一除以 1000：
# y_tr_mm, y_val_mm, y_test_mm = y_tr_mm/1000, y_val_mm/1000, y_test_mm/1000
# y_pred_tr_mm, y_pred_val_mm, y_pred_test_mm = y_pred_tr_mm/1000, y_pred_val_mm/1000, y_pred_test_mm/1000

# ===== 5) 指标 =====
def evaluate(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2   = r2_score(y_true, y_pred)
    return rmse, r2

tr_rmse, tr_r2   = evaluate(y_tr_mm,   y_pred_tr_mm)
val_rmse, val_r2 = evaluate(y_val_mm,  y_pred_val_mm)
te_rmse, te_r2   = evaluate(y_test_mm, y_pred_test_mm)

print(f"XGBoost Train RMSE: {tr_rmse:.4f} mm/6h \tR²: {tr_r2:.4f}")
print(f"XGBoost Val   RMSE: {val_rmse:.4f} mm/6h \tR²: {val_r2:.4f}")
print(f"XGBoost Test  RMSE: {te_rmse:.4f} mm/6h \tR²: {te_r2:.4f}")